Celda 1: Importación de librerías y carga de datos limpios

In [5]:
import sys
!{sys.executable} -m pip install scipy scikit-learn imbalanced-learn


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import levene
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler, SMOTENC
 
PATH_LIMPIO = r"D:\Funeraria_Inventario_Inteligente_porqueria\data\processed\dataset\dataset_limpio.xlsx"
PATH_MENSUAL = r"D:\Funeraria_Inventario_Inteligente_porqueria\data\processed\dataset\dataset_mensual.xlsx"
PATH_OUTPUT_DIR = r"D:\Funeraria_Inventario_Inteligente_porqueria\data\processed\augmented"
 
os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)
 
df_registro = pd.read_excel(PATH_LIMPIO)
df_mensual = pd.read_excel(PATH_MENSUAL)
 
print(f"Dataset registro: {len(df_registro)} registros")
print(f"Dataset mensual: {len(df_mensual)} meses")

Dataset registro: 340 registros
Dataset mensual: 46 meses


In [23]:
def bootstrap_temporal(df, n_iter=5, ruido_monto=50, ruido_dias=3):
    df_bootstrap = df.copy()
    df_bootstrap['origen'] = 'original'
 
    for i in range(n_iter):
        df_synthetic = df.sample(n=len(df), replace=True, random_state=42 + i).copy()
        df_synthetic['Monto'] = df_synthetic['Monto'] + np.random.normal(0, ruido_monto, len(df_synthetic))
        df_synthetic['Monto'] = df_synthetic['Monto'].clip(lower=0)
        df_synthetic['Fecha'] = df_synthetic['Fecha'] + pd.to_timedelta(
            np.random.randint(-ruido_dias, ruido_dias + 1, len(df_synthetic)), unit='D'
        )
        df_synthetic['origen'] = f'bootstrap_{i+1}'
        df_bootstrap = pd.concat([df_bootstrap, df_synthetic], ignore_index=True)
 
    return df_bootstrap
 
df_bootstrap = bootstrap_temporal(df_registro)
print(f"Bootstrapping completado: {len(df_registro)} -> {len(df_bootstrap)} registros")
print(df_bootstrap['origen'].value_counts())

Bootstrapping completado: 340 -> 2040 registros
origen
original       340
bootstrap_1    340
bootstrap_2    340
bootstrap_3    340
bootstrap_4    340
bootstrap_5    340
Name: count, dtype: int64


In [24]:
def filtrar_clases_raros(df, min_registros=5):
    conteos = df['Ataud_Modelo'].value_counts()
    clases_validas = conteos[conteos >= min_registros].index.tolist()
    n_antes = df['Ataud_Modelo'].nunique()
    df_filtrado = df[df['Ataud_Modelo'].isin(clases_validas)].copy()
    n_despues = df_filtrado['Ataud_Modelo'].nunique()
    print(f"Filtrado: {n_antes} -> {n_despues} clases (se eliminaron {n_antes - n_despues} clases con <{min_registros} registros)")
    return df_filtrado
 
df_registro_filtrado = filtrar_clases_raros(df_registro, min_registros=5)

Filtrado: 86 -> 11 clases (se eliminaron 75 clases con <5 registros)


In [25]:
print(df_registro_filtrado['Monto'].describe())
print("NaN en Monto:", df_registro_filtrado['Monto'].isna().sum())
print(df_registro_filtrado['Monto'].sort_values().tail(10))  # outliers altos
print(df_registro_filtrado['Monto'].sort_values().head(10))  # outliers bajos o ceros

count       245.000000
mean       8423.391837
std       57407.934997
min         180.000000
25%        2070.000000
50%        2300.000000
75%        3200.000000
max      870000.000000
Name: Monto, dtype: float64
NaN en Monto: 9
105    870000.0
20          NaN
38          NaN
92          NaN
112         NaN
127         NaN
168         NaN
213         NaN
314         NaN
316         NaN
Name: Monto, dtype: float64
104    180.0
17     200.0
72     200.0
263    250.0
107    450.0
48     600.0
22     600.0
161    800.0
315    890.0
233    950.0
Name: Monto, dtype: float64


In [26]:
print(df_registro_filtrado.loc[105])

Fecha                     2024-02-01 00:00:00
Forma de pago                         directo
Ataud_Modelo                         Imperial
Ataud_Color                   no_especificado
Capilla                           sin_capilla
Carroza                                     0
Carroza flores                              0
Cargadores                                  0
Auto                                        0
Microbus                                    0
Monto                                870000.0
fecha_sospechosa                        False
monto_outlier                            True
Monto_winsorizado                    154600.0
Ataud_completo       Imperial_no_especificado
Periodo                               2024-02
Name: 105, dtype: object


In [27]:
def aplicar_smote(df, columnas_continuas, columnas_categoricas, target_col='Ataud_Modelo', k_neighbors=3):
    """
    Aplica SMOTENC para generar registros sintéticos balanceados por clase.
 
    - columnas_continuas: variables numéricas continuas (ej. Monto_winsorizado)
    - columnas_categoricas: variables discretas/binarias (ej. Carroza, Auto, Cargadores),
      tratadas como categóricas por SMOTENC para que NO se interpolen a decimales.
 
    Las demás columnas (no usadas por SMOTENC) se reconstruyen para las filas
    sintéticas mediante muestreo con reemplazo de los valores originales.
    """
    df_work = df.copy().reset_index(drop=True)
 
    columnas_numericas = columnas_continuas + columnas_categoricas
 
    # Imputar NaN en columnas numéricas con la mediana antes de aplicar SMOTENC
    for col in columnas_numericas:
        if df_work[col].isna().any():
            n_nan = df_work[col].isna().sum()
            mediana = df_work[col].median()
            df_work[col] = df_work[col].fillna(mediana)
            print(f"  -> Columna '{col}': {n_nan} NaN imputados con la mediana ({mediana})")
 
    # Codificar la clase objetivo
    le_target = LabelEncoder()
    y = le_target.fit_transform(df_work[target_col])
 
    # Verificar que cada clase tenga al menos k_neighbors+1 muestras
    conteos = pd.Series(y).value_counts()
    k_eff = min(k_neighbors, conteos.min() - 1)
    k_eff = max(k_eff, 1)
 
    # Orden de columnas pasadas a SMOTENC: primero continuas, luego categóricas
    X = df_work[columnas_continuas + columnas_categoricas].copy()
 
    # Índices (posiciones) de las columnas categóricas dentro de X
    categorical_indices = list(range(len(columnas_continuas), len(columnas_continuas) + len(columnas_categoricas)))
 
    smotenc = SMOTENC(categorical_features=categorical_indices, sampling_strategy='auto',
                      k_neighbors=k_eff, random_state=42)
    X_res, y_res = smotenc.fit_resample(X, y)
 
    df_result = pd.DataFrame(X_res, columns=columnas_continuas + columnas_categoricas)
 
    # Las columnas categóricas vuelven a su tipo original (entero) tras SMOTENC
    for col in columnas_categoricas:
        df_result[col] = df_result[col].round().astype(df_work[col].dtype if pd.api.types.is_integer_dtype(df_work[col]) else int)
 
    df_result[target_col] = le_target.inverse_transform(y_res)
 
    # Marcar origen: las primeras len(df_work) filas son originales
    n_original = len(df_work)
    df_result['origen'] = 'original'
    df_result.loc[n_original:, 'origen'] = 'smote_sintetico'
 
    # Para las columnas no usadas por SMOTENC, rellenar las filas
    # sintéticas muestreando con reemplazo de los valores originales
    otras_cols = [c for c in df_work.columns if c not in columnas_numericas + [target_col]]
    for col in otras_cols:
        valores_originales = df_work[col].values
        n_sint = len(df_result) - n_original
        if n_sint > 0:
            indices_muestra = np.random.choice(len(valores_originales), size=n_sint, replace=True)
            muestras = valores_originales[indices_muestra]
            columna_completa = np.concatenate([valores_originales, muestras])
        else:
            columna_completa = valores_originales
        df_result[col] = pd.Series(columna_completa, index=df_result.index)
 
    return df_result, k_eff
 
 
# Columna continua: Monto_winsorizado (ya tratado para outliers) en lugar de Monto crudo
columna_monto = 'Monto_winsorizado' if 'Monto_winsorizado' in df_registro_filtrado.columns else 'Monto'
columnas_continuas_smote = [c for c in [columna_monto] if c in df_registro_filtrado.columns]
 
# Columnas categóricas/discretas: NO se interpolan a decimales con SMOTENC
columnas_categoricas_smote = [c for c in ['Carroza', 'Auto', 'Cargadores'] if c in df_registro_filtrado.columns]
 
columnas_numericas_smote = columnas_continuas_smote + columnas_categoricas_smote
 
df_smote, k_usado = aplicar_smote(
    df_registro_filtrado,
    columnas_continuas_smote,
    columnas_categoricas_smote,
    target_col='Ataud_Modelo'
)
 
print(f"\nSMOTE completado (k_neighbors={k_usado}): {len(df_registro_filtrado)} -> {len(df_smote)} registros")
print("Distribución de clases después de SMOTE:")
print(df_smote['Ataud_Modelo'].value_counts())

  -> Columna 'Monto_winsorizado': 9 NaN imputados con la mediana (2300.0)

SMOTE completado (k_neighbors=3): 254 -> 836 registros
Distribución de clases después de SMOTE:
Ataud_Modelo
Imperial     76
sin_ataud    76
Principe     76
Biblia       76
Americano    76
Parvulo      76
Lincoln      76
Redondo      76
Modelo       76
Madera       76
Diamante     76
Name: count, dtype: int64


In [28]:
def test_levene_variables(df_original, df_sintetico, columnas, alpha=0.05):
    """
    Aplica el Test de Levene a cada columna numérica para comparar
    la homogeneidad de varianzas entre el dataset original y el
    dataset sintético (aumentado).
 
    H0: las varianzas de ambos grupos son iguales (homocedasticidad)
    H1: las varianzas son diferentes (heterocedasticidad)
    """
    resultados = []
    for col in columnas:
        original_vals = df_original[col].dropna().astype(float)
        sintetico_vals = df_sintetico[col].dropna().astype(float)
 
        stat, p_value = levene(original_vals, sintetico_vals)
 
        resultados.append({
            'Variable': col,
            'Media_Original': original_vals.mean(),
            'Media_Sintetico': sintetico_vals.mean(),
            'Var_Original': original_vals.var(),
            'Var_Sintetico': sintetico_vals.var(),
            'Levene_Statistic': stat,
            'p_value': p_value,
            'Homocedasticidad (p>=alpha)': p_value >= alpha,
            'Conclusion': 'Varianzas homogéneas (no se rechaza H0)' if p_value >= alpha
                          else 'Varianzas distintas (se rechaza H0)'
        })
 
    return pd.DataFrame(resultados)
 
 
# Subconjunto del sintético generado por SMOTE
df_smote_sintetico_only = df_smote[df_smote['origen'] == 'smote_sintetico']
 
df_levene_smote = test_levene_variables(
    df_registro_filtrado,
    df_smote_sintetico_only,
    columnas_numericas_smote
)
 
print("\n=== Test de Levene: Original vs SMOTE sintético ===")
print(df_levene_smote.to_string(index=False))
 
# También validar el bootstrap temporal (Monto)
df_bootstrap_sintetico_only = df_bootstrap[df_bootstrap['origen'] != 'original']
df_levene_bootstrap = test_levene_variables(
    df_registro,
    df_bootstrap_sintetico_only,
    ['Monto']
)
 
print("\n=== Test de Levene: Original vs Bootstrap sintético ===")
print(df_levene_bootstrap.to_string(index=False))


=== Test de Levene: Original vs SMOTE sintético ===
         Variable  Media_Original  Media_Sintetico  Var_Original  Var_Sintetico  Levene_Statistic  p_value  Homocedasticidad (p>=alpha)                              Conclusion
Monto_winsorizado     5420.126531      2964.865416  3.057668e+08   3.674687e+07          8.023040 0.004731                        False     Varianzas distintas (se rechaza H0)
          Carroza        0.850394         0.838488  1.277271e-01   1.356590e-01          0.188099 0.664616                         True Varianzas homogéneas (no se rechaza H0)
             Auto        0.287402         0.336770  2.056114e-01   2.237403e-01          1.974735 0.160319                         True Varianzas homogéneas (no se rechaza H0)
       Cargadores        2.850394         2.962199  3.321403e+00   3.079464e+00          0.915722 0.338878                         True Varianzas homogéneas (no se rechaza H0)

=== Test de Levene: Original vs Bootstrap sintético ===
Variable  

In [29]:
sns.set_style("whitegrid")
 
# 5.1 Boxplots comparativos: Original vs SMOTE sintético, por variable
fig, axes = plt.subplots(1, len(columnas_numericas_smote), figsize=(5 * len(columnas_numericas_smote), 5))
if len(columnas_numericas_smote) == 1:
    axes = [axes]
 
for ax, col in zip(axes, columnas_numericas_smote):
    data_plot = pd.concat([
        pd.DataFrame({col: df_registro_filtrado[col], 'Dataset': 'Original'}),
        pd.DataFrame({col: df_smote_sintetico_only[col], 'Dataset': 'SMOTE sintético'})
    ], ignore_index=True)
    sns.boxplot(data=data_plot, x='Dataset', y=col, ax=ax, palette='Set2')
    ax.set_title(f'Distribución: {col}')
 
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT_DIR, 'levene_boxplots_smote.png'), dpi=150)
plt.close()
print("\nGráfico guardado: levene_boxplots_smote.png")
 
# 5.2 Histogramas/KDE comparativos
fig, axes = plt.subplots(1, len(columnas_numericas_smote), figsize=(5 * len(columnas_numericas_smote), 5))
if len(columnas_numericas_smote) == 1:
    axes = [axes]
 
for ax, col in zip(axes, columnas_numericas_smote):
    sns.kdeplot(df_registro_filtrado[col], ax=ax, label='Original', fill=True, alpha=0.4)
    sns.kdeplot(df_smote_sintetico_only[col], ax=ax, label='SMOTE sintético', fill=True, alpha=0.4)
    ax.set_title(f'Densidad: {col}')
    ax.legend()
 
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT_DIR, 'levene_densidad_smote.png'), dpi=150)
plt.close()
print("Gráfico guardado: levene_densidad_smote.png")
 
# 5.3 Barras: estadístico de Levene y p-value por variable
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
axes[0].bar(df_levene_smote['Variable'], df_levene_smote['Levene_Statistic'], color='steelblue')
axes[0].set_title('Estadístico de Levene por variable')
axes[0].set_ylabel('Estadístico')
axes[0].tick_params(axis='x', rotation=30)
 
axes[1].bar(df_levene_smote['Variable'], df_levene_smote['p_value'], color='darkorange')
axes[1].axhline(0.05, color='red', linestyle='--', label='alpha = 0.05')
axes[1].set_title('p-value del Test de Levene')
axes[1].set_ylabel('p-value')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()
 
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT_DIR, 'levene_resultados_smote.png'), dpi=150)
plt.close()
print("Gráfico guardado: levene_resultados_smote.png")
 
# 5.4 Comparación de medias y varianzas (barras agrupadas)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(df_levene_smote))
width = 0.35
 
axes[0].bar(x - width/2, df_levene_smote['Media_Original'], width, label='Original', color='steelblue')
axes[0].bar(x + width/2, df_levene_smote['Media_Sintetico'], width, label='SMOTE sintético', color='darkorange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_levene_smote['Variable'], rotation=30)
axes[0].set_title('Comparación de medias')
axes[0].legend()
 
axes[1].bar(x - width/2, df_levene_smote['Var_Original'], width, label='Original', color='steelblue')
axes[1].bar(x + width/2, df_levene_smote['Var_Sintetico'], width, label='SMOTE sintético', color='darkorange')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_levene_smote['Variable'], rotation=30)
axes[1].set_title('Comparación de varianzas')
axes[1].legend()
 
plt.tight_layout()
plt.savefig(os.path.join(PATH_OUTPUT_DIR, 'levene_medias_varianzas_smote.png'), dpi=150)
plt.close()
print("Gráfico guardado: levene_medias_varianzas_smote.png")

C:\Users\capmd\AppData\Local\Temp\ipykernel_1164\2162059157.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=data_plot, x='Dataset', y=col, ax=ax, palette='Set2')
C:\Users\capmd\AppData\Local\Temp\ipykernel_1164\2162059157.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=data_plot, x='Dataset', y=col, ax=ax, palette='Set2')
C:\Users\capmd\AppData\Local\Temp\ipykernel_1164\2162059157.py:13: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=data_plot, x='Dataset', y=col, ax=ax, palette='Set2')
C:\Users\capmd\AppData\Local\Te


Gráfico guardado: levene_boxplots_smote.png
Gráfico guardado: levene_densidad_smote.png
Gráfico guardado: levene_resultados_smote.png
Gráfico guardado: levene_medias_varianzas_smote.png


In [30]:
def crear_sliding_windows(df_mensual, window_size=3, targets=None):
    if targets is None:
        targets = ['servicios_totales', 'monto_total']
 
    df_win = df_mensual.copy()
    if 'Periodo' in df_win.columns:
        df_win = df_win.set_index('Periodo')
 
    windows = []
    for i in range(window_size, len(df_win)):
        window = {}
        for target in targets:
            if target in df_win.columns:
                for lag in range(1, window_size + 1):
                    window[f'{target}_lag_{lag}'] = df_win[target].iloc[i - lag]
                window[f'{target}_target'] = df_win[target].iloc[i]
        window['mes_target'] = str(df_win.index[i])
        windows.append(window)
 
    return pd.DataFrame(windows)
 
targets_disponibles = [c for c in ['servicios_totales', 'monto_total'] if c in df_mensual.columns]
df_windows = crear_sliding_windows(df_mensual, window_size=3, targets=targets_disponibles)
 
print(f"\nSliding windows generadas: {len(df_windows)}")


Sliding windows generadas: 43


In [31]:
def generar_reporte(df_orig, df_smote, df_bootstrap, df_windows, df_levene_smote, df_levene_bootstrap):
    reporte = []
    reporte.append(('registros_originales', len(df_orig)))
    reporte.append(('registros_smote_total', len(df_smote)))
    reporte.append(('registros_smote_sinteticos', len(df_smote) - len(df_orig)))
    reporte.append(('distribucion_clases_antes', str(df_orig['Ataud_Modelo'].value_counts().to_dict())))
    reporte.append(('distribucion_clases_despues', str(df_smote['Ataud_Modelo'].value_counts().to_dict())))
    reporte.append(('registros_bootstrap_total', len(df_bootstrap)))
    reporte.append(('ventanas_generadas', len(df_windows)))
    reporte.append(('window_size', 3))
 
    for _, row in df_levene_smote.iterrows():
        reporte.append((f"levene_pvalue_SMOTE_{row['Variable']}", row['p_value']))
        reporte.append((f"levene_homocedasticidad_SMOTE_{row['Variable']}", row['Homocedasticidad (p>=alpha)']))
 
    for _, row in df_levene_bootstrap.iterrows():
        reporte.append((f"levene_pvalue_BOOTSTRAP_{row['Variable']}", row['p_value']))
        reporte.append((f"levene_homocedasticidad_BOOTSTRAP_{row['Variable']}", row['Homocedasticidad (p>=alpha)']))
 
    return pd.DataFrame(reporte, columns=['Metrica', 'Valor'])
 
reporte = generar_reporte(df_registro_filtrado, df_smote, df_bootstrap, df_windows, df_levene_smote, df_levene_bootstrap)
print("\nReporte final:")
print(reporte.to_string(index=False))


Reporte final:
                                        Metrica                                                                                                                                                                     Valor
                           registros_originales                                                                                                                                                                       254
                          registros_smote_total                                                                                                                                                                       836
                     registros_smote_sinteticos                                                                                                                                                                       582
                      distribucion_clases_antes     {'Americano': 76, 'Lincoln': 38, 'Imperial': 26, 'sin_ataud'

In [33]:
df_smote.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_smote.xlsx'), index=False)
df_bootstrap.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_bootstrap.xlsx'), index=False)
df_windows.to_excel(os.path.join(PATH_OUTPUT_DIR, 'dataset_sliding_window.xlsx'), index=False)
reporte.to_excel(os.path.join(PATH_OUTPUT_DIR, 'reporte_augmentation.xlsx'), index=False)
df_levene_smote.to_excel(os.path.join(PATH_OUTPUT_DIR, 'levene_smotec.xlsx'), index=False)
df_levene_bootstrap.to_excel(os.path.join(PATH_OUTPUT_DIR, 'levene_bootstrap.xlsx'), index=False)
 
print(f"\nArchivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de augmentation y validación completado.")


Archivos exportados en: D:\Funeraria_Inventario_Inteligente_porqueria\data\processed\augmented
Pipeline de augmentation y validación completado.
